# Addestrare BeatNet-CRNN su Colab (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alessandro1040/bpm-crnn/blob/main/notebooks/03_training_colab.ipynb)

Addestra il modello da zero con la stessa pipeline del repo e valuta il
risultato. Su una **T4** il training e' molto piu' veloce che su un portatile
(M5: 29 min per 60 epoche, ~38 campioni/s).

Cosa fa:
0. dipendenze + verifica GPU;
1. clone del repo;
2. dati (`scripts/prepare_data.sh`: GTZAN 1.2 GB + annotazioni + cache Mel);
3. training con `src.train` (stessi iperparametri del modello pubblicato,
   parametri modificabili: `EPOCHS`, `BATCH`, `SYNTHETIC`);
4. valutazione sul test set + confronto con il modello rilasciato;
5. export dei pesi per l'inferenza (21 MB) e download.

**Attenzione**: il disco di Colab e' effimero. Se la sessione cade perdi
`runs/` - scarica il `.pt` alla fine (cella 5) o monta Google Drive.

In [ ]:
#@title 0. Dipendenze e GPU
%pip install -q torch librosa soundfile pandas matplotlib tqdm requests

import subprocess
import sys
from pathlib import Path

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("Nessuna GPU: vai su Runtime > Change runtime type > GPU (T4)")

print("torch", torch.__version__)

In [ ]:
#@title 1. Clona il repo (e i pesi rilasciati)
import json, os, subprocess, sys, urllib.request
from pathlib import Path

import numpy as np
import torch

REPO_URL = "https://github.com/Alessandro1040/bpm-crnn.git"
ROOT = Path("/content/bpm-crnn") if Path("/content").is_dir() else Path.cwd() / "bpm-crnn"
if not (ROOT / "src" / "infer.py").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

CKPT = ROOT / "models" / "bpm_crnn_gtzan.pt"
RAW_URL = "https://raw.githubusercontent.com/Alessandro1040/bpm-crnn/main/models/bpm_crnn_gtzan.pt"
RELEASE_URL = "https://github.com/Alessandro1040/bpm-crnn/releases/download/v1.0-gtzan/best.pth"
if not CKPT.exists():
    try:
        print("scarico i pesi da GitHub (models/bpm_crnn_gtzan.pt, 21 MB) ...")
        urllib.request.urlretrieve(RAW_URL, CKPT)
    except Exception as exc:                       # fallback: release asset (64 MB, con optimizer)
        print("fallback sulla release:", exc)
        urllib.request.urlretrieve(RELEASE_URL, CKPT)
print("pesi:", CKPT, f"{CKPT.stat().st_size/1e6:.1f} MB")

In [ ]:
#@title 2. Dati: GTZAN + annotazioni + cache Mel
# GTZAN (1.2 GB) + annotazioni di tempo + metadati/split/cache Mel.
# Salta i passi gia' fatti (utile se rilanci la cella).
r = subprocess.run(["bash", "scripts/prepare_data.sh"], cwd=ROOT)
print("\nprepare_data -> exit", r.returncode)

In [ ]:
#@title 3. Training (stessa ricetta del modello pubblicato)
EPOCHS = 60          # il modello pubblicato usa 60 epoche
BATCH = 64           # su una T4 ci sta comodo; su CPU usa 16-32
SYNTHETIC = 500      # loop percussivi sintetici con BPM esatto per epoca

cmd = [sys.executable, "-m", "src.train",
       "--out", "runs/colab",
       "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH),
       "--eval-batch-size", "128",
       "--synthetic", str(SYNTHETIC),
       "--num-workers", "2"]
print(" ".join(cmd), "\n")
subprocess.run(cmd, cwd=ROOT, check=True)
print("\n--- stato finale del run ---")
print((ROOT / "runs/colab/status.json").read_text())

In [ ]:
#@title 4. Valutazione e confronto col modello rilasciato
import json

# 4a. valutazione sul test set
subprocess.run([sys.executable, "-m", "src.evaluate",
                "--ckpt", "runs/colab/best.pth",
                "--cache-dir", "data/cache",
                "--out-dir", "runs/colab",
                "--plots-dir", "assets",
                "--batch-size", "64"], cwd=ROOT, check=True)

# 4b. confronto con il modello rilasciato
nuovo = json.loads((ROOT / "runs/colab/test_metrics.json").read_text())
vecchio = json.loads((ROOT / "results/gtzan/test_metrics.json").read_text())
print(f"\n{'metrica':<24}{'questo run':>12}{'rilasciato':>12}")
print("-" * 48)
for key in ("MAE", "MedianAE", "Acc_1%", "Acc_5%", "Acc_octave_4%", "P_score"):
    scala = 100.0 if key.startswith("Acc") else 1.0
    print(f"{key:<24}{scala * nuovo['mean/' + key]:12.2f}"
          f"{scala * vecchio['mean/' + key]:12.2f}")

In [ ]:
#@title 5. Salva i pesi addestrati
# 5. export dei pesi di inferenza (21 MB) + download
subprocess.run([sys.executable, "scripts/export_inference.py",
                "--ckpt", "runs/colab/best.pth",
                "--out", "models/bpm_crnn_colab.pt",
                "--metrics", "runs/colab/test_metrics.json"], cwd=ROOT, check=True)

try:
    from google.colab import files
    files.download(str(ROOT / "models/bpm_crnn_colab.pt"))
except Exception as exc:
    print("download manuale:", exc)
print("per pubblicarlo su GitHub: Settings > Releases > Draft a new release, "
      "allega il .pt (oppure `gh release create` con un token).")

### Note

* Con una T4 bastano pochi minuti per 60 epoche: puoi permetterti
  `EPOCHS = 200` (il cosine annealing e' impostato su `--epochs`, quindi cambia
  anche lo schedule) e `SYNTHETIC = 2000`.
* Il collo di bottiglia qui e' la generazione dei loop sintetici (fatta nei
  worker del DataLoader): piu' `--num-workers` hai, meglio e' (Colab free: 2 vCPU).
* Colab base ha un limite di ~12 h di sessione e nessuna persistenza su disco:
  per run lunghi monta Drive (`from google.colab import drive; drive.mount('/content/drive')`)
  e metti `--out /content/drive/MyDrive/bpm-crnn/runs/colab`.
* I numeri attesi sono quelli del README (MAE ~15.8 BPM sul test GTZAN con 60
  epoche): il dataset e' piccolo e le annotazioni rumorose, quindi non aspettarti
  "MAE < 2 BPM" senza un corpus piu' grande (Ballroom / Extended Ballroom / FMA).